## Create Black masks file

In [1]:
import cv2
import numpy as np
import os

# 1. Point this EXACTLY to where the Br35H 'no' images are
image_dir = 'figshare_braintumor/images/no_tumor/' 

# 2. Point this to where the masks should go
mask_output_dir = 'figshare_braintumor/masks/no_tumor/'

os.makedirs(mask_output_dir, exist_ok=True)

# Now the loop will find the .jpg files inside that specific folder
for img_name in os.listdir(image_dir):
    img_path = os.path.join(image_dir, img_name)
    
    # Skip if it's a subfolder accidentally
    if os.path.isdir(img_path):
        continue
        
    img = cv2.imread(img_path)
    if img is not None:
        black_mask = np.zeros(img.shape[:2], dtype=np.uint8)
        cv2.imwrite(os.path.join(mask_output_dir, img_name), black_mask)

print(f"Success! Generated {len(os.listdir(mask_output_dir))} black masks.")

Success! Generated 1500 black masks.


In [ ]:
# import os
# import shutil
# from collections import defaultdict
# from sklearn.model_selection import train_test_split

# def parse_filename(filename):
#     # String splitting is much safer and more flexible than regex here
#     name, ext = os.path.splitext(filename)
    
#     if ext.lower() not in ['.jpg', '.jpeg', '.png', '.tif']:
#         return None
        
#     parts = name.split('_')
#     if len(parts) >= 3:
#         global_id = parts[0]
#         label = parts[-1]
#         # Joining everything in the middle handles patient IDs with underscores
#         patient_id = "_".join(parts[1:-1]) 
#         return global_id, patient_id, label
        
#     return None

# # ── Step 1: Collect all files grouped by patient_id ──────────────────────────
# patient_files = defaultdict(list)
# skipped_files = [] # <-- Diagnostic tracker

# images_root = "figshare_braintumor/images"
# masks_root  = "figshare_braintumor/masks"

# for label_folder in sorted(os.listdir(images_root)):          
#     label_dir = os.path.join(images_root, label_folder)
    
#     if not os.path.isdir(label_dir):
#         continue
        
#     for fname in os.listdir(label_dir):
#         parsed = parse_filename(fname)
#         if not parsed:
#             skipped_files.append(f"FAILED PARSING: {fname}")
#             continue
            
#         global_num, patient_id, label = parsed
#         img_path  = os.path.join(images_root, label_folder, fname)
#         mask_path = os.path.join(masks_root,  label_folder, fname)
        
#         # Double check the mask actually exists!
#         if os.path.exists(mask_path):
#             patient_files[patient_id].append((img_path, mask_path, label))
#         else:
#             skipped_files.append(f"MISSING MASK: {fname}")

# # --- Diagnostic Print ---
# total_valid_files = sum(len(v) for v in patient_files.values())
# print(f"Successfully loaded {total_valid_files} valid image/mask pairs.")

# if skipped_files:
#     print(f"\nSkipped {len(skipped_files)} files. Here are a few examples:")
#     for skipped in skipped_files[:10]:
#         print(f" - {skipped}")

# # ── Step 2: Split patient IDs (stratify by majority label per patient) ────────
# if total_valid_files > 0:
#     patient_ids = list(patient_files.keys())
#     patient_labels = [patient_files[pid][0][2] for pid in patient_ids]

#     train_pids, test_pids = train_test_split(
#         patient_ids,
#         test_size=0.2,
#         stratify=patient_labels,   
#         random_state=42
#     )

#     train_pids, val_pids = train_test_split(
#         train_pids,
#         test_size=0.1,
#         stratify=[patient_files[pid][0][2] for pid in train_pids],
#         random_state=42
#     )

#     print(f"\nPatients → Train: {len(train_pids)}, Val: {len(val_pids)}, Test: {len(test_pids)}")
#     assert not set(train_pids) & set(test_pids), "LEAKAGE DETECTED"
#     assert not set(val_pids)   & set(test_pids), "LEAKAGE DETECTED"

# # ── Step 3: Copy files into the new structure ─────────────────────────────────
# def copy_split(pid_list, split_name, base_out="figshare_braintumor_split"):
#     # Note: Changed base_out so it doesn't write into the same directory it's reading from
#     for pid in pid_list:
#         for img_path, mask_path, label in patient_files[pid]:
#             for src, kind in [(img_path, "images"), (mask_path, "masks")]:
#                 dst_dir = os.path.join(base_out, split_name, kind, label)
#                 os.makedirs(dst_dir, exist_ok=True)
#                 shutil.copy2(src, dst_dir)

# print("\nCopying files...")
# copy_split(train_pids, "train")
# copy_split(val_pids,   "val")
# copy_split(test_pids,  "test")
# print("Done!")

Successfully loaded 3064 valid image/mask pairs.

Skipped 1500 files. Here are a few examples:
 - FAILED PARSING: no0.jpg
 - FAILED PARSING: no1.jpg
 - FAILED PARSING: no10.jpg
 - FAILED PARSING: no100.jpg
 - FAILED PARSING: no1000.jpg
 - FAILED PARSING: no1001.jpg
 - FAILED PARSING: no1002.jpg
 - FAILED PARSING: no1003.jpg
 - FAILED PARSING: no1004.jpg
 - FAILED PARSING: no1005.jpg

Patients → Train: 167, Val: 19, Test: 47

Copying files...
Done!


## Change parse_filename

In [ ]:
import os
import shutil
from collections import defaultdict
from sklearn.model_selection import train_test_split

def parse_filename(filename):
    name, ext = os.path.splitext(filename)
    
    if ext.lower() not in ['.jpg', '.jpeg', '.png', '.tif']:
        return None
    
    # NEW LOGIC: Handle Br35H "No Tumor" files
    # FIND THIS PART IN YOUR CODE:
    if name.startswith('no'):
        return {
        'global_id': name,
        'patient_id': name,  # CHANGE THIS from 'healthy_set' to 'name'
        'label': 'no_tumor'
    }
        
    # Original logic for Figshare files
    parts = name.split('_')
    if len(parts) >= 3:
        return {
            'global_id': parts[0],
            'patient_id': "_".join(parts[1:-1]),
            'label': parts[-1]
        }
    
    return None

# ── Step 1: Collect all files grouped by patient_id ──────────────────────────
patient_files = defaultdict(list)
skipped_files = [] # <-- Diagnostic tracker

images_root = "figshare_braintumor/images"
masks_root  = "figshare_braintumor/masks"

for label_folder in sorted(os.listdir(images_root)):          
    label_dir = os.path.join(images_root, label_folder)
    
    if not os.path.isdir(label_dir):
        continue
        
    # --- UPDATE THIS BLOCK IN YOUR LOOP ---
    for fname in os.listdir(label_dir):
        parsed = parse_filename(fname)
        if not parsed:
            skipped_files.append(f"FAILED PARSING: {fname}")
            continue
        
        # Extract values correctly from the dictionary
        global_num = parsed['global_id']
        patient_id = parsed['patient_id']
        label      = parsed['label']
        
        img_path  = os.path.join(images_root, label_folder, fname)
        mask_path = os.path.join(masks_root,  label_folder, fname)
    
    # ... rest of your code ...
        
        # Double check the mask actually exists!
        if os.path.exists(mask_path):
            patient_files[patient_id].append((img_path, mask_path, label))
        else:
            skipped_files.append(f"MISSING MASK: {fname}")

# --- Diagnostic Print ---
total_valid_files = sum(len(v) for v in patient_files.values())
print(f"Successfully loaded {total_valid_files} valid image/mask pairs.")

if skipped_files:
    print(f"\nSkipped {len(skipped_files)} files. Here are a few examples:")
    for skipped in skipped_files[:10]:
        print(f" - {skipped}")

# ── Step 2: Split patient IDs (stratify by majority label per patient) ────────
if total_valid_files > 0:
    patient_ids = list(patient_files.keys())
    patient_labels = [patient_files[pid][0][2] for pid in patient_ids]

    train_pids, test_pids = train_test_split(
        patient_ids,
        test_size=0.2,
        stratify=patient_labels,   
        random_state=42
    )

    train_pids, val_pids = train_test_split(
        train_pids,
        test_size=0.1,
        stratify=[patient_files[pid][0][2] for pid in train_pids],
        random_state=42
    )

    print(f"\nPatients → Train: {len(train_pids)}, Val: {len(val_pids)}, Test: {len(test_pids)}")
    assert not set(train_pids) & set(test_pids), "LEAKAGE DETECTED"
    assert not set(val_pids)   & set(test_pids), "LEAKAGE DETECTED"

# ── Step 3: Copy files into the new structure ─────────────────────────────────
def copy_split(pid_list, split_name, base_out="figshare_braintumor_split"):
    # Note: Changed base_out so it doesn't write into the same directory it's reading from
    for pid in pid_list:
        for img_path, mask_path, label in patient_files[pid]:
            for src, kind in [(img_path, "images"), (mask_path, "masks")]:
                dst_dir = os.path.join(base_out, split_name, kind, label)
                os.makedirs(dst_dir, exist_ok=True)
                shutil.copy2(src, dst_dir)

print("\nCopying files...")
copy_split(train_pids, "train")
copy_split(val_pids,   "val")
copy_split(test_pids,  "test")
print("Done!")

Successfully loaded 4564 valid image/mask pairs.

Patients → Train: 1247, Val: 139, Test: 347

Copying files...
Done!


## Create Black masks 